# EEG Power Classifier: Frontal Theta & Posterior Alpha

This notebook trains a classifier using EEG power features:
- **Frontal Theta** (4-8 Hz at Fz/FCz): Marker of focused attention meditation
- **Posterior Alpha** (8-12 Hz at Pz/O1/O2): Reflects reduced visual/sensory processing

Classification: med1breath (Label 1) vs think1 (Label 0)

## 1. Setup

In [5]:
import numpy as np
import os
from pathlib import Path
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

env_path = Path('.').resolve() / '.env'
load_dotenv(env_path)

from eeg_processor import Config, DataLoader, EEGPowerExtractor, HRVClassifier

print(f"BIDS Root: {os.getenv('BIDS_ROOT')}")

BIDS Root: d:\Hackathons\EPFL_Life_Sciences_2026\ds003969


## 2. Configuration

In [6]:
config = Config(bids_root=os.getenv('BIDS_ROOT'), random_seed=42)

MEDITATE_TASK = 'med1breath'  # Label 1
THINK_TASK = 'think1'         # Label 0
MAX_SUBJECTS = 10

print(f"Tasks: {MEDITATE_TASK} (label 1) vs {THINK_TASK} (label 0)")

[OK] Configuration initialized with BIDS root: d:\Hackathons\EPFL_Life_Sciences_2026\ds003969
Tasks: med1breath (label 1) vs think1 (label 0)


## 3. Load Data

In [7]:
loader = DataLoader(config)
loader.explore_bids_structure()

loaded_data = {}
for subject_id in loader.subjects[:MAX_SUBJECTS]:
    print(f"Subject {subject_id}:")
    raw_med = loader.load_eeg_data(subject_id, session=None, task=MEDITATE_TASK)
    raw_think = loader.load_eeg_data(subject_id, session=None, task=THINK_TASK)
    
    if raw_med and raw_think:
        loaded_data[subject_id] = {MEDITATE_TASK: raw_med, THINK_TASK: raw_think}
        print(f"  ✓ Both tasks loaded")

print(f"\nLoaded {len(loaded_data)} subjects")

[OK] BIDS structure explored successfully
  - Subjects: 1 (['001'])
  - Sessions: 0 (None)
  - Tasks: 4 (['med1breath', 'med2', 'think1', 'think2'])
Subject 001:
Loading BIDS file: d:/Hackathons/EPFL_Life_Sciences_2026/ds003969/sub-001/eeg/sub-001_task-med1breath_eeg.bdf
[OK] Successfully loaded EEG data


d:\Hackathons\EPFL_Life_Sciences_2026\EEG-Meditation-App\eeg\eeg_processor\data_loader.py:93: RuntimeWarning: Did not find any events.tsv associated with sub-001_task-med1breath.

The search_str was "d:\Hackathons\EPFL_Life_Sciences_2026\ds003969\sub-001\**\eeg\sub-001*events.tsv"
  )
d:\Hackathons\EPFL_Life_Sciences_2026\EEG-Meditation-App\eeg\eeg_processor\data_loader.py:93: RuntimeWarning: The number of channels in the channels.tsv sidecar file (79) does not match the number of channels in the raw data file (80). Will not try to set channel names.
  )
d:\Hackathons\EPFL_Life_Sciences_2026\EEG-Meditation-App\eeg\eeg_processor\data_loader.py:93: RuntimeWarning: Unable to map the following column(s) to to MNE:
gender: m
group: htr
ethnicity: indian
first_session: meditation
sleep: 6
education: 0
years_of_practice: 3
notes: n/a
  )


  - Shape: (80, 620544)
Loading BIDS file: d:/Hackathons/EPFL_Life_Sciences_2026/ds003969/sub-001/eeg/sub-001_task-think1_eeg.bdf
[OK] Successfully loaded EEG data


d:\Hackathons\EPFL_Life_Sciences_2026\EEG-Meditation-App\eeg\eeg_processor\data_loader.py:93: RuntimeWarning: Did not find any events.tsv associated with sub-001_task-think1.

The search_str was "d:\Hackathons\EPFL_Life_Sciences_2026\ds003969\sub-001\**\eeg\sub-001*events.tsv"
  )
d:\Hackathons\EPFL_Life_Sciences_2026\EEG-Meditation-App\eeg\eeg_processor\data_loader.py:93: RuntimeWarning: The number of channels in the channels.tsv sidecar file (79) does not match the number of channels in the raw data file (80). Will not try to set channel names.
  )
d:\Hackathons\EPFL_Life_Sciences_2026\EEG-Meditation-App\eeg\eeg_processor\data_loader.py:93: RuntimeWarning: Unable to map the following column(s) to to MNE:
gender: m
group: htr
ethnicity: indian
first_session: meditation
sleep: 6
education: 0
years_of_practice: 3
notes: n/a
  )


  - Shape: (80, 620544)
  ✓ Both tasks loaded

Loaded 1 subjects


## 4. Extract EEG Power Features

In [10]:
all_features = []
all_labels = []
feature_names = ['frontal_theta', 'posterior_alpha']
channel_mapping = {'EXG1': 'eog', 'EXG2': 'eog', 'EXG3': 'eog', 'EXG4': 'eog',
                   'EXG5': 'misc', 'EXG6': 'misc', 'EXG7': 'ecg'}

extractor = EEGPowerExtractor(sampling_rate=500)

for subject_id, tasks_data in loaded_data.items():
    print(f"Subject {subject_id}:")
    
    for task_name, raw in tasks_data.items():
        # Setup channels
        loader.raw = raw
        loader.setup_montage(montage_name='biosemi64', on_missing='ignore')
        loader.set_channel_types(channel_mapping)
        
        # Extract power features
        try:
            features, _ = extractor.extract_from_raw(raw)
            print(f"  ✓ {task_name}: theta={features['frontal_theta']:.2f}, alpha={features['posterior_alpha']:.2f}")
            all_features.append([features['frontal_theta'], features['posterior_alpha']])
            all_labels.append(1 if task_name == MEDITATE_TASK else 0)
        except Exception as e:
            print(f"  ✗ {task_name}: {e}")

X = np.array(all_features)
y = np.array(all_labels, dtype=int)

print(f"\nExtracted {len(X)} samples")
print(f"Shape: {X.shape}")
print(f"Classes: Think={np.sum(y==0)}, Meditate={np.sum(y==1)}")

Subject 001:
[OK] Montage 'biosemi64' set successfully
[OK] Channel types updated: 7 channels
  ✓ med1breath: theta=0.00, alpha=0.00
[OK] Montage 'biosemi64' set successfully
[OK] Channel types updated: 7 channels
  ✓ think1: theta=0.00, alpha=0.00

Extracted 2 samples
Shape: (2, 2)
Classes: Think=1, Meditate=1


## 5. Data Split

In [9]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

ValueError: The least populated classes in y have only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2. Classes with too few members are: [0, 1]

## 6. Train Model

In [ ]:
classifier = HRVClassifier(input_size=2, hidden_sizes=[32, 16], learning_rate=0.001)

print(f"Training with {X_train.shape[1]} features...\n")
history = classifier.train(X_train, y_train, X_val, y_val, epochs=100, batch_size=8, feature_names=feature_names)

## 7. Training Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='Train', linewidth=2)
axes[1].plot(history['val_acc'], label='Val', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluate

In [ ]:
test_pred, test_probs = classifier.predict_with_proba(X_test)

accuracy = accuracy_score(y_test, test_pred)
precision = precision_score(y_test, test_pred)
recall = recall_score(y_test, test_pred)
f1 = f1_score(y_test, test_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

## 9. Predictions

In [ ]:
print("Sample predictions with probabilities:\n")
for i in range(min(5, len(X_test))):
    theta, alpha = X_test[i]
    pred = "Think" if test_pred[i] == 0 else "Meditate"
    true = "Think" if y_test[i] == 0 else "Meditate"
    print(f"Sample {i+1}: theta={theta:.2f}, alpha={alpha:.2f}")
    print(f"  True: {true}, Pred: {pred}, Think: {test_probs[i][0]:.3f}, Med: {test_probs[i][1]:.3f}\n")

## 10. Save Model

In [ ]:
model_dir = Path('.').resolve() / 'models' / 'eeg_power_classifier'
classifier.save(str(model_dir))
print(f"Model saved to {model_dir}")